In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torchvision import transforms
from torch.utils.data import DataLoader

current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

sys.path.append(str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
splits_path = PROJECT_ROOT / "data" / "metadata" / "splits.csv"
split_df = pd.read_csv(splits_path)

identity_ids = sorted(split_df["identity_id"].unique())

identity_to_label = {identity_id: index for index, identity_id in enumerate(identity_ids)}
split_df["label_idx"] = split_df["identity_id"].map(identity_to_label)

num_classes = len(identity_to_label)

In [3]:
from src.dataset import PigReIDDataset

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

query_df = split_df[split_df["pool_status"] == "query"].copy()
gallery_df = split_df[split_df["pool_status"] == "gallery"].copy()

query_dataset = PigReIDDataset(
    dataframe=query_df,
    transform=train_transform
)

gallery_dataset = PigReIDDataset(
    dataframe=gallery_df,
    transform=train_transform
)

query_loader = DataLoader(
    query_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

gallery_loader = DataLoader(
    gallery_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Query size:", len(query_dataset))
print("Gallery size:", len(gallery_dataset))

Query size: 24
Gallery size: 72


In [4]:
from src.evaluation import extract_embeddings, compute_rank1_map

def evaluate_model(model):
    query_embeddings, query_labels, query_identity_ids, query_image_paths = extract_embeddings(
        model=model,
        dataloader=query_loader,
        device=device
    )
    
    gallery_embeddings, gallery_labels, gallery_identity_ids, gallery_image_paths = extract_embeddings(
        model=model,
        dataloader=gallery_loader,
        device=device
    )
    
    metrics = compute_rank1_map(
        query_embeddings=query_embeddings,
        query_labels=query_labels,
        gallery_embeddings=gallery_embeddings,
        gallery_labels=gallery_labels
    )
    
    return metrics

d:\anaconda\Lib\site-packages\torchmetrics\__init__.py:31: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.1)
  import scipy.signal


In [5]:
RANDOM_SEED = 42

RANDOM_BUDGET = 60

initial_train_df = split_df[split_df["pool_status"] == "initial_labeled"].copy()

unlabeled_df = split_df[split_df["pool_status"] == "unlabeled"].copy()

random_selected_df = unlabeled_df.sample(
    n=RANDOM_BUDGET,
    random_state=RANDOM_SEED
).copy()

random_selected_df["pool_status"] = "random_labeled"

random_selected_df["is_labeled"] = True

random_train_df = pd.concat(
    [initial_train_df, random_selected_df],
    ignore_index=True
)

print("Initial train size:", len(initial_train_df))
print("Random selected size:", len(random_selected_df))
print("Random train size:", len(random_train_df))

Initial train size: 72
Random selected size: 60
Random train size: 132


In [6]:
from src.training import train_reid_model

model_random, history_random = train_reid_model(
    train_df=random_train_df,
    train_transform=train_transform,
    num_classes=num_classes,
    device=device,
    epochs=10,
    batch_size=16,
    learning_rate=1e-4,
    embedding_dim=256,
    seed=42
)

Epoch 1/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 1: total_loss=2.0734, ce_loss=1.7812, triplet_loss=0.2922


Epoch 2/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 2: total_loss=2.0033, ce_loss=1.7274, triplet_loss=0.2759


Epoch 3/10:   0%|          | 0/9 [00:00<?, ?it/s]

...

Epoch 8: total_loss=1.5870, ce_loss=1.4904, triplet_loss=0.0966


Epoch 9/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 9: total_loss=1.6107, ce_loss=1.4907, triplet_loss=0.1200


Epoch 10/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 10: total_loss=1.5436, ce_loss=1.4635, triplet_loss=0.0801


In [7]:
metrics_random = evaluate_model(model_random)
print(f"Random Rank-1: {metrics_random['rank1']:.4f}")
print(f"Random mAP: {metrics_random['mAP']:.4f}")

Random Rank-1: 0.3333
Random mAP: 0.3523
